# closures
**Prerequisites:** function_basics, parameters_and_arguments, return_values, scope

**Outcome:** After this notebook you will know exactly what a closure is at the memory level, how inner functions capture variables from enclosing scope, and how to use closures to build stateful functions, factories, and configurable behaviour without classes.

## The Problem

In [ ]:
# you need three validators with different thresholds
# writing them separately is repetitive

def is_low_latency(value):
    return value < 100

def is_medium_latency(value):
    return value < 300

def is_acceptable_latency(value):
    return value < 500

# the only difference is the threshold number
# three functions, three maintenance points
# what if you need 10 thresholds?
print(is_low_latency(80))
print(is_medium_latency(250))
print(is_acceptable_latency(490))

## The Concept

A closure is a function that remembers the variables from the enclosing scope in which it was created, even after that scope has finished executing. When an inner function references a variable from an outer function, Python does not copy the value — it keeps a live reference to the variable cell. That cell persists as long as the inner function exists. This means you can create functions that carry private state, configure behaviour at creation time, and behave differently from each other despite sharing the same code.

## Minimal Example

In [ ]:
def make_threshold_check(threshold):
    def check(value):
        return value < threshold    # threshold lives in enclosing scope
    return check

is_low        = make_threshold_check(100)
is_medium     = make_threshold_check(300)
is_acceptable = make_threshold_check(500)

print(is_low(80))          # True
print(is_medium(250))      # True
print(is_acceptable(490))  # True
print(is_low(150))         # False

## How Python Does It

When a function references a name from an enclosing scope Python promotes that variable to a cell object — a wrapper that holds the actual value. Both the outer function and the inner function share a pointer to the same cell. When the outer function finishes its local frame is destroyed but the cell object survives because the inner function still holds a reference to it. You can inspect this with `__closure__` and `cell_contents`.

In [ ]:
def make_threshold_check(threshold):
    def check(value):
        return value < threshold
    return check

is_low = make_threshold_check(100)

# inspect the closure
print(is_low.__closure__)                        # tuple of cell objects
print(is_low.__closure__[0].cell_contents)       # 100 — the captured value
print(is_low.__code__.co_freevars)               # ('threshold',) — names captured

## Building Up

In [ ]:
# closure with multiple captured variables
def make_range_check(low, high):
    def check(value):
        return low <= value <= high
    return check

is_normal_latency = make_range_check(50, 200)
is_normal_cpu     = make_range_check(10, 80)

print(is_normal_latency(120))   # True
print(is_normal_latency(300))   # False
print(is_normal_cpu(75))        # True

In [ ]:
# closure as a stateful counter — no class needed
def make_counter(start=0, step=1):
    count = start

    def increment():
        nonlocal count
        count += step
        return count

    def reset():
        nonlocal count
        count = start

    def current():
        return count

    return increment, reset, current

inc, reset, cur = make_counter(start=0, step=5)

print(inc())    # 5
print(inc())    # 10
print(inc())    # 15
print(cur())    # 15
reset()
print(cur())    # 0

In [ ]:
# closure as a function factory — configure once, call many times
def make_prefixer(prefix):
    def add_prefix(text):
        return f"{prefix}: {text}"
    return add_prefix

info    = make_prefixer("INFO")
warning = make_prefixer("WARN")
error   = make_prefixer("ERROR")

print(info("pipeline started"))
print(warning("disk usage above 80%"))
print(error("db connection failed"))

In [ ]:
# each closure instance is independent — separate cells
def make_accumulator():
    total = 0
    def add(value):
        nonlocal total
        total += value
        return total
    return add

acc_a = make_accumulator()
acc_b = make_accumulator()

print(acc_a(10))   # 10
print(acc_a(20))   # 30
print(acc_b(5))    # 5  — independent from acc_a
print(acc_a(5))    # 35 — acc_a unaffected by acc_b

In [ ]:
# partial application using closures
def make_retry_runner(max_retries):
    def run(job_id):
        for attempt in range(1, max_retries + 1):
            print(f"  attempt {attempt}/{max_retries} for {job_id}")
        print(f"  {job_id} exhausted retries")
    return run

run_with_3 = make_retry_runner(3)
run_with_5 = make_retry_runner(5)

run_with_3("job_1")
print()
run_with_5("job_2")

## Where It Breaks

In [ ]:
# classic loop closure bug — all closures capture the same cell
checks = []
for threshold in [100, 200, 300]:
    def check(value):
        return value < threshold   # threshold is a loop variable — one shared cell
    checks.append(check)

# by the time we call these, the loop has finished and threshold == 300
print(checks[0](150))   # expected True  (< 100) — got True  (< 300) — wrong reason
print(checks[1](250))   # expected False (< 200) — got True  (< 300) — wrong
print(checks[2](350))   # expected False (< 300) — got False (< 300) — right by accident

# all three closures share the same cell and see threshold = 300

In [ ]:
# confirming the bug — all cells point to the same value
print([c.__closure__[0].cell_contents for c in checks])   # [300, 300, 300]

## The Fix

In [ ]:
# fix 1: use a default argument to capture the value at creation time
checks = []
for threshold in [100, 200, 300]:
    def check(value, t=threshold):   # t is bound to current threshold immediately
        return value < t
    checks.append(check)

print(checks[0](150))   # False — correctly uses 100
print(checks[1](150))   # True  — correctly uses 200
print(checks[2](150))   # True  — correctly uses 300

In [ ]:
# fix 2: use a factory function — cleaner, more explicit
def make_check(threshold):
    def check(value):
        return value < threshold
    return check

checks = [make_check(t) for t in [100, 200, 300]]

print(checks[0](150))   # False
print(checks[1](150))   # True
print(checks[2](150))   # True

## In a Real System

In [ ]:
# a validation pipeline built from closures
# each validator is configured once and reused across many records

def make_required_field_check(field):
    def check(record):
        return bool(record.get(field))
    check.__name__ = f"has_{field}"
    return check

def make_max_value_check(field, limit):
    def check(record):
        return record.get(field, 0) <= limit
    check.__name__ = f"{field}_lte_{limit}"
    return check

def validate(record, checks):
    failures = [c.__name__ for c in checks if not c(record)]
    return {"ok": len(failures) == 0, "failures": failures}

validators = [
    make_required_field_check("id"),
    make_required_field_check("status"),
    make_max_value_check("retries", 3),
]

records = [
    {"id": "job_1", "status": "pending", "retries": 2},
    {"id": "job_2", "status": "",        "retries": 1},
    {"id": "",      "status": "done",    "retries": 5},
]

for record in records:
    result = validate(record, validators)
    print(f"{record.get('id') or 'no-id':8} ok={result['ok']}  failures={result['failures']}")

## Performance

Creating a closure allocates one cell object per captured variable plus the inner function object itself — a small, fixed cost. Calling a closure adds one level of indirection to access the cell contents compared to a plain local variable. In practice this overhead is unmeasurable in any non-trivial workload. The memory cost is the cell surviving as long as any closure referencing it is alive — be aware of this in long-running processes that create large numbers of closures capturing large objects.

## Anti-Pattern

In [ ]:
# anti-pattern: using a closure to hide state that should be explicit
def make_job_processor():
    processed = []

    def process(job_id):
        processed.append(job_id)   # hidden state, impossible to inspect from outside
        return f"{job_id} done"

    return process

processor = make_job_processor()
processor("job_1")
processor("job_2")
# how many jobs have been processed? you cannot tell from outside

# correct: return all state you need, or expose it explicitly
def make_job_processor():
    processed = []

    def process(job_id):
        processed.append(job_id)
        return f"{job_id} done"

    def get_processed():
        return list(processed)    # expose state explicitly

    return process, get_processed

process, get_processed = make_job_processor()
process("job_1")
process("job_2")
print(get_processed())   # ["job_1", "job_2"] — inspectable

## Interview Signals

- What is a closure and what problem does it solve?
- What is a cell object and why does Python use it to implement closures?
- What is the loop closure bug and what are two ways to fix it?
- When would you use a closure instead of a class?

## Exercise

In [ ]:
def make_rate_limiter(max_calls, period_label):
    """
    Returns a function call_allowed() that:
    - Returns True if the number of calls so far is below max_calls
    - Returns False once max_calls has been reached
    - Never resets (no time-based logic needed)

    Also returns a function get_stats() that returns a dict:
        {'calls': <int>, 'limit': <int>, 'period': <str>}

    Use closures only — no classes, no global variables.
    Bug: the function bodies are missing — implement them.
    """
    pass


call_allowed, get_stats = make_rate_limiter(3, "per_minute")

assert call_allowed()  == True
assert call_allowed()  == True
assert call_allowed()  == True
assert call_allowed()  == False
assert call_allowed()  == False

stats = get_stats()
assert stats["calls"]  == 5,            f"got {stats['calls']}"
assert stats["limit"]  == 3,            f"got {stats['limit']}"
assert stats["period"] == "per_minute", f"got {stats['period']}"

# two rate limiters must not share state
call_b, stats_b = make_rate_limiter(2, "per_second")
call_b()
assert stats_b()["calls"] == 1
assert get_stats()["calls"] == 5   # original unchanged

print("all assertions passed")